<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_WorldFactCollector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

In [1]:
!pip -q install pymupdf
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py                  # all
!wget -O mos1.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos1.py                  # all
import mos, mos1
from pathlib import Path
import time

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 13.5 MB/s eta 0:00:00


In [2]:
import importlib
#importlib.reload(mos1)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

WORLDFACTS = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldFacts"

!ls -l "{WORLDFACTS}"

Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF


## Static World Data
URLs stored as PDF files that are converted to markdown files

In [4]:
client = mos.authenticateOpenAI()

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


In [ ]:
# =============================================================================
# MOS STATIC SOURCE PACK v1
# =============================================================================

# 01. IEA — Oil Market Report, September 2026
# Global oil supply, demand, inventories, prices, refining and Gulf disruption.
# https://www.iea.org/reports/oil-market-report-september-2026

# 02. US EIA — Short-Term Energy Outlook: Global Oil Markets
# Oil prices, Middle East production/exports, Iran and Hormuz disruption.
# https://www.eia.gov/outlooks/steo/report/global_oil.php

# 03. PPAC — India's Oil & Gas Ready Reckoner / Current Reports
# India's crude production, imports, consumption, refining and import dependence.
# https://ppac.gov.in/
#Snapshot of India's Oil and Gas Data → Monthly Ready Reckoner — August 2026 → View Report (PDF)

# 04. Government of India / PIB — West Asia Energy Supply Disruption
# India's Gulf/Hormuz exposure and government energy-security response.
# https://www.pib.gov.in/PressReleasePage.aspx?PRID=2239021&lang=1&reg=6


# 06. World Bank — Iran Macro Poverty Outlook, April 2026
# Concise current picture of Iran's economy, sanctions, inflation, fiscal stress and oil vulnerability.
# https://documents.worldbank.org/en/publication/documents-reports/documentdetail/099211508192619381
# https://documents1.worldbank.org/curated/en/099211508192619381/pdf/Bulk-8dc686b3-acf3-4bd1-92b1-373dd5db8389.pdf

# 07. SIPRI — Yearbook 2026 Summary
# Regional strategic context, military capabilities and nuclear forces.
# https://www.sipri.org/yearbook/2026/summary/sipri-yearbook-2026-summary-english

# 08. CSIS Missile Threat — Iran
# Iranian ballistic/cruise missiles, anti-ship systems and UAV capabilities.
# https://missilethreat.csis.org/country/iran/

# =============================================================================

In [ ]:
# 01. US Congressional Research Service (CRS) — U.S. Conflict with Iran
# Comprehensive overview of the 2026 US-Iran conflict, military operations, Hormuz, sanctions, diplomacy, regional actors and strategic issues.
# https://www.congress.gov/crs-product/R48887

# 02. UK House of Commons Library — Iran in 2026: Six Months After the US/Israeli Strikes
# Broad current assessment of the conflict, Iran's political/economic condition, Hormuz, sanctions, regional armed groups and US-Iran diplomacy.
# https://commonslibrary.parliament.uk/research-briefings/cbp-11075/

In [5]:
PDF_DIR = f"{WORLDFACTS}/PDF"
MD_DIR  = f"{WORLDFACTS}/MD"

!ls -l "{PDF_DIR}"

total 20688
-rw------- 1 root root   153646 Sep 28 21:39  Iran_Data.pdf
-rw------- 1 root root  7057875 Sep 28 21:41 'Missiles of Iran _ Missile Threat.pdf'
-rw------- 1 root root   214516 Sep 28 03:04 'Oil Market Report - September 2026 – Analysis - IEA.pdf'
-rw------- 1 root root   259068 Sep 28 21:32 'Press Release Page _ Press Information Bureau.pdf'
-rw------- 1 root root   381243 Sep 28 21:28 'Short-Term Energy Outlook - U.S. Energy Information Administration (EIA).pdf'
-rw------- 1 root root  1605684 Sep 28 21:40  SIPRI_Summary.pdf
-rw------- 1 root root 10479230 Sep 29 12:53  Snapshot_of_Indias_Oil_Gas_data.pdf
-rw------- 1 root root   175382 Sep 29 12:51 'UK-Commons-Iran in 2026_ Six months after the US_Israeli strikes.pdf'
-rw------- 1 root root   855750 Sep 29 12:51  US-Congress-R48887.3.pdf


In [6]:
Role_PDFReader = """
    You are a document-cleaning engine.

    Your task is to convert raw text extracted from a PDF into
    clean, well-structured Markdown.

    RULES

    1. Preserve ALL substantive factual content.
    2. Preserve all numbers, dates, quantities, percentages,
       names, units and factual claims.
    3. Preserve meaningful headings and section structure.
    4. Preserve useful tables. Reconstruct them as Markdown
       tables where reasonably possible.
    5. Remove advertisements, navigation menus, subscription
       prompts, cookie notices, social-media links, repeated
       headers and footers, page furniture and similar junk.
    6. Repair obvious line-break, hyphenation and formatting
       damage introduced by PDF extraction.
    7. Do NOT summarize.
    8. Do NOT interpret.
    9. Do NOT add information.
    10. Do NOT use outside knowledge.
    11. Do NOT correct or reconcile factual claims.
    12. If something is genuinely unclear or corrupted,
        preserve it rather than inventing content.
    13. If date of publication of document is known, state this at the top as date of publication.
        If date of publicatino of document is not clear, state date of publication not known. Do not guess

    Return ONLY the cleaned Markdown document.
    """

In [7]:
from pathlib import Path

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))

# ------------------------------------------------------------------
# Delete old Markdown files
# ------------------------------------------------------------------

old_md_files = list(Path(MD_DIR).glob("*.md"))

for md_file in old_md_files:
    md_file.unlink()

print(f"Deleted {len(old_md_files)} old MD files")
print(f"Found {len(pdf_files)} PDF files\n")

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files\n")

md_files = []
for source_PDF in pdf_files:
    print("=" * 70)
    print(f"Processing: {source_PDF.name}")
    try:
        md_path = mos1.GetWorldFile(client,Role_PDFReader,source_PDF,MD_DIR, _model="gpt-5-mini" )
        md_files.append(md_path)
    except Exception as e:
        print(f"❌ FAILED: {source_PDF.name}")
        print(f"Reason: {e}")

print("\n" + "=" * 70)
print(f"Completed: {len(md_files)} of {len(pdf_files)} files")

Deleted 7 old MD files
Found 9 PDF files

Found 9 PDF files

Processing: Iran_Data.pdf
Reading  : Iran_Data.pdf
Extracted: 5,504 characters
Created  : IranData_260929.md
Markdown : 7,755 characters
Processing: Missiles of Iran _ Missile Threat.pdf
Reading  : Missiles of Iran _ Missile Threat.pdf
Extracted: 5,594 characters
Created  : MissilesofIranMissil_260929.md
Markdown : 6,571 characters
Processing: Oil Market Report - September 2026 – Analysis - IEA.pdf
Reading  : Oil Market Report - September 2026 – Analysis - IEA.pdf
Extracted: 7,853 characters
Created  : OilMarketReportSepte_260929.md
Markdown : 8,373 characters
Processing: Press Release Page _ Press Information Bureau.pdf
Reading  : Press Release Page _ Press Information Bureau.pdf
Extracted: 12,271 characters
Created  : PressReleasePagePres_260929.md
Markdown : 12,237 characters
Processing: SIPRI_Summary.pdf
Reading  : SIPRI_Summary.pdf
Extracted: 76,446 characters
Created  : SIPRISummary_260929.md
Markdown : 76,689 character

In [8]:
#print(md_files)

In [9]:
for f in md_files:
    print(f.name)

IranData_260929.md
MissilesofIranMissil_260929.md
OilMarketReportSepte_260929.md
PressReleasePagePres_260929.md
SIPRISummary_260929.md
ShortTermEnergyOutlo_260929.md
SnapshotofIndiasOilG_260929.md
UKCommonsIranin2026S_260929.md
USCongressR488873_260929.md


In [10]:
mos1.ShowMD(md_files[7])

# SOURCE: UK-Commons-Iran in 2026_ Six months after the US_Israeli strikes

    **Source PDF:** `UK-Commons-Iran in 2026_ Six months after the US_Israeli strikes.pdf`

    ---

    Date of publication: 28 August 2026

# Iran in 2026: Six months after the US/Israeli strikes

There are no talks between the US and Iran on ending the conflict. The US is imposing more economic sanctions. Iran’s high use of the death penalty persists.

28 August 2026 marks six months since the US and Israel began a series of strikes against Iran, saying they were targeting Iranian military infrastructure and its nuclear and ballistic missile programme. They also called for a change of leadership.

Iran responded to the strikes by launching a series of attacks on neighbouring Arab states and announcing it would close the Strait of Hormuz. The US announced a counter-blockade of Iranian ports in the Persian Gulf. Both blockades remain in place, and only a limited number of ships continue to traverse the strait.

The parties to the conflict agreed a memorandum of understanding (MOU) after Pakistani-led mediation in June, with the intention to facilitate talks over a 60-day period. However, fighting continued after it was agreed. Both Iran and the US have said there are no current plans to restart talks.

This briefing describes the situation in Iran, six months into the conflict. While Iran has a new Supreme Leader, there are some divisions within ruling groups. The regime also continues to act against those engaged in the January 2026 protests. The US has launched new sanctions, targeting Iran’s already strained economy. Iran’s allies in the region are under pressure to disarm.

The Commons Library collection, Iran in 2026, provides further information on the Pakistan-mediated talks, the launch of military operations, attempts to reopen the Strait of Hormuz, and the UK Government response since February.

---

## What is the status of Iran’s leadership?

Iran’s Supreme Leader, Ali Khamenei, was killed in the airstrikes on 29 February 2026. His son, Mojtaba Khameni, was selected as his successor. He has not been seen in public since his appointment.

While there is little public information on dynamics within Iran’s Government, some analysts have suggested that Iran has moved towards a shared leadership model during the conflict and that the role of the Islamic Revolutionary Guard Corps’ (IRGC) has increased. The IRGC is a major military and security force operating both within Iran and abroad (including in the UK).

### Public disagreement on the June MOU

Within Iran, there continue to be public divisions on future talks with the United States. Since the MOU was made in June, some Iranian MPs have argued it was agreed against the Supreme Leader’s negotiating position, while in August the Assembly of Experts (which appoints the supreme leader) said that it was an “illusion” to trust the United States.

The Iranian President, Masoud Pezeshkian, has continued to publicly back the June MOU, saying in August that “it is better to end the war today”. He has denied a rift with the Supreme Leader on the issue. In June, Mojtaba Khamenei had said he had taken a “different view” on the MOU but had received sufficient reassurances from the President.

### New security official and his conditions on Hormuz

Analysis for Foreign Policy and the Gulf States Newsletter have also suggested that appointment of Mohsen Rezaei as secretary of Iran’s Supreme National Security Council (which helps set foreign and defence policy) in August means that Iran may be preparing for a longer conflict. Major General Razaei is close to the Supreme Leader and formerly of the IRGC.

On his appointment, Major General Razaei told Iranian media that the Strait of Hormuz would only be reopened if six conditions are met. These include:

- withdrawal of US forces near to Iran,
- removal of the US naval blockade,
- release of Iran’s frozen assets overseas, and
- a “forever” end to conflict in Gaza, Iraq, Lebanon and Yemen (these theatres were not all explicitly mentioned in the June MOU).

While there are no current US–Iran talks, on 25 August Pakistan reported “significant progress” in its engagement with Iran.

---

## What is the status of Iran-backed groups in the region?

Iran has long maintained a network of armed and terrorist groups in the Middle East, which aim to compete with or threaten regional powers (notably Saudi Arabia or Israel), establish friendly governments, and export Iran’s Islamic revolution to other countries. Many groups have their own separate networks and base of support beyond Iran.

During the conflicts with Israel from 2023 to 2025 and in 2026, the military capacity of many of these groups was substantially degraded. From 2025, there has been greater pressure for three groups (Hamas, Hezbollah, and the Popular Mobilisation Forces in Iraq) to disarm.

According to US military assessments, US/Israeli strikes in 2026 have substantially degraded Iran’s ability to project power (PDF) across the region using its own armed forces.

### Hamas in Gaza

Under the October 2025 framework between Israel and Hamas, Hamas, which the UK has proscribed as a terrorist organisation, is to disarm and no longer govern Gaza (among other provisions).

In August 2026, in a statement welcomed by the United States and UK Government, Hamas agreed to a plan that includes the phased surrender of its weapons. Israel initially rejected the plan, but later said it would engage on disarmament talks.

Among the issues are the timing of disarmament and Israeli withdrawal. Analysis for the Chatham House think tank argues “it is unlikely that a timeline for decommissioning weapons or drawing down [Israeli] troops [in Gaza] will be agreed anytime soon”.

To verify the disarmament process and replace Hamas’ security forces, the establishment of International Stabilisation Force for Gaza is required and the Palestinian National Committee would have to be allowed into Gaza (it is currently in Egypt). Both bodies are established under the 2025 framework, and separate to the Palestinian Authority in the West Bank.

### The Popular Mobilisation Forces in Iraq

The Popular Mobilisation Forces (PMF) are a set of legalised paramilitary groups operating in Iraq, some of which are supported by Iran. The new Iraqi Government that took office in 2026 aims to disarm or integrate the Popular Mobilisation Forces (PMF) into the state armed forces by the end of September 2026 (the same date as US military withdrawal from Iraq).

As of August, a draft law is to be considered by Iraq’s parliament, though several PMF groups have rejected the plan.

Analysis for Foreign Policy magazine notes disarmament is not immediately on the horizon, but the “most plausible course” in the short term is that the state will gain greater control over elements of the PMF and isolate the “factions that will continue to pursue an independent regional agenda” (namely those aligned with Iran).

### Hezbollah in Lebanon

Hezbollah is a non-state armed group in Lebanon, which the UK has proscribed as a terrorist organisation. Lebanon’s Government had set a deadline to disarm Hezbollah, by the end of 2025.

While this was not achieved, in 2026 the Lebanese Government has banned Hezbollah’s military activity and extended the territorial control of the Lebanese Armed Forces (LAF) into areas previously held by Hezbollah.

Despite US mediation in 2026 between Lebanon and Israel after the Israel–Hezbollah conflict earlier in the year, Hezbollah has rejected disarmament. Israel says Hezbollah must first disarm before it withdraws from Lebanon.

Analysis for the Center for Strategic and International Studies argues that the US-mediated agreement to facilitate Israeli withdrawal and LAF return offers a “significant opportunity” for the Lebanese Government to restore state sovereignty and achieve a state monopoly on weapons.

### The Houthis in Yemen

The Houthis continue to control most of Yemen’s population. In 2025, UN observers reported the group had “increased their combat readiness and preparedness” (PDF), while in 2026 the US intelligence community assessment said the group “remain[ed] a resilient challenger” to the US and others.

While there has not been a return to nationwide conflict in Yemen since the end of Yemen’s 2022 ceasefire, in 2026 the Houthis have renewed their attacks on Saudi Arabia (which backs the Houthi’s main opponents, the internationally recognised government of Yemen) and said the Red Sea is closed to Saudi shipping. In response, Saudi Arabia has established a naval coalition and conducted some air strikes in Yemen.

Analysis for Armed Conflict and Location Event Data (ACLED) judges the renewed conflict in 2026 to be limited so far. The Houthis have framed their action as being in “response to Saudi violations only”, rather than in wider support of Iran (analysis echoed by the Arab Gulf States Institute in Washington).

---

## Arrests and execution rates in Iran remain high

Extensive nationwide protests occurred in Iran from late December 2025 to mid-January 2026. In response, authorities severely restricted internet access and there was violent repression of protests; rights groups estimated that several thousand people were killed in state violence. International internet access was not restored until the end of May.

The protests were initially sparked by economic challenges (see below) but later expanded to include some political demands. Analysts considered them to be an extensive set of protests, but unlikely to result in the regime’s overthrow.

The Commons Library briefing on the protests has more information.

### Arrests and new security laws

State repression and arrests in response to the protests has continued, joined by further restrictions in response to the 2026 conflict.

Since the beginning of the US/Israel strikes on 28 February, Iranian security forces have arrested several thousand people, accusing them of being “collaborators” with the US and Israel, and spies.

This follows new laws to expand the scope of national security crimes (including against the use of satellite equipment and some types of information sharing) and increasing the number of crimes subject to the death penalty. Iran’s parliament is also considering new laws to criminalise or limit engagement with foreign media.

The security forces have particularly targeted areas populated by minority ethnic groups, according to human rights groups.

### Execution rates

In August 2026, the UN High Commissioner for Human Rights Volker Türk said that since March at least 56 people has been executed on national-security-related charges, including 27 people linked to the protests.

As set out in Amnesty’s annual reports on the death penalty, Iran has long had one of the world’s highest execution rates. Minority ethnic groups in Iran are overrepresented in death penalty statistics, according to data collected by the Norway-based Iran Human Rights.

In August 2026, the UK issued a joint statement alongside 32 governments condemning the “use of capital punishment to silence dissent, intimidate communities, and punish individuals”.

There is also ongoing debate on the enforcement of hijab (Islamic head dress) with some Iranian reformists saying that the state’s demands that people wear it are causing internal division.

---

## Iran’s inflation rate is high and economic challenges remain

In August 2026, Iran’s currency reached a record low against the US dollar and the year-on-year inflation rate reached a record 84% rise.

The Iranian President has acknowledged that there are “many problems in society”, including inflation and unemployment.

### Iran’s economy has struggled for several years

As set out in section 1 of the Commons Library’s introductory country profile on Iran, before the 2026 conflict began Iran was already experiencing high levels of inflation, low levels of job creation, and water and energy challenges. Economic sanctions, particularly from the US, were one reason for this.

These economic challenges have been exacerbated by the 2026 conflict, Iran’s closure of the Strait of Hormuz and the US counter-blockade of Iranian ports (though there was some recovery in the initial stages of the June MOU, when US sanctions were lessened and Iranian oil exports resumed).

In July, the International Monetary Fund projected Iran would experience growth rates of –5.4% in 2026 and +2.9% in 2027 (if the conflict ends).

### Damage from the conflict and energy shortages

The degree of economic damage from the 2026 conflict is uncertain, though Iran was seeking US$270 billion in damages during talks with the US.

Damage has been reported to gas production, industrial locations and power sites. There has been a loss of trade routes through Hormuz (through which most Iranian trade flows) and a rise in unemployment (partially also caused by the internet shutdown earlier in the year).

Iran has also struggled to meet domestic petrol demand, and rolling energy blackouts have continued (having begun before the conflict).

### Iranian capacity to absorb economic shocks

The Speaker of the Iranian Parliament, Mohammad Baqer Qalibaf, has acknowledged that, “no matter how much military power we have […] we will not endure” if “people are hungry and we do not have financial circulation, economic growth and national production”.

The existing US blockade on Iranian ports has caused a substantial fall in oil exports, meaning Iran has been unable to capitalise on higher oil prices. Up to a third of the Iranian state budget comes from oil revenue.

However, a CIA analysis seen by the Washington Post newspaper in May 2026 estimates that Iran is unlikely to face economic “hardship” for several more months.

Foreign Policy analysis in August also notes Iran has a long history of adapting its economy to sanctions.

---

## New US economic sanctions campaign launched

In mid-2026, the US has shifted towards using economic pressure to secure concessions from Iran. In August, the US President said the US would conduct “economic warfare and isolation on an unprecedented scale” to “cripple” Iran and stop it developing a nuclear weapon.

Iran has long been under extensive US economic sanctions, banning almost all US trade with the country as well as targeting sectors like oil. These sanctions include the “maximum pressure” campaign, first launched by the first Trump administration in 2018 after its withdrawal from the 2015 nuclear deal, and which was renewed under the second Trump administration in 2025.

### Announcement of expansion of US sanctions

On 24 August, the Trump administration announced “Operation Economic Outcast”. It said that an expanded US sanctions regime would target several Iranian sectors, such as technology, oil, gold and shipping (including Iran’s “shadow fleet”), as well as financial mechanisms used to evade sanctions and fund the IRGC and armed and terrorist groups abroad.

The US Treasury Secretary, Scott Bessent, said that US economic pressure would make it “likely there will not be a large-scale kinetic [military] restart”.

### The US will also target countries trading with Iran

In its announcement, the Trump administration said that “every country” would be “given a defined timeline to shut down the Iran-related activity” that the US has identified.

If the deadline is not met, the US Treasury Secretary said those states would be “hasten[ing] the[ir] economic ostracism”.

The US has not set out what the potential economic measures for countries trading for Iran may be.

Following the announcement, the UK Government said it would “continue working alongside the US and other partners to apply economic pressure on Iran” and would not allow Iran to “abuse” the UK financial system.

### China is one of Iran’s main trading partners

No countries trading with Iran were named by the United States when the sanctions operation was launched. However, while it was not named, an estimated 90% of recent Iranian oil exports have been to China.

Media reports expect the US to apply further sanctions on Iranian trade with China. China has said it “firmly opposes” what it has described as “illegal unilateral sanctions” (referring to those imposed by individual countries without the endorsement of the UN Security Council).

Reuters and Al Jazeera have summaries of Iran’s major trading partners.

### The UAE also plans to cut off economic ties

On 19 August, the United Arab Emirates (UAE) said it would cut off all trade and commercial and financial transactions with Iran, citing recent Iranian attacks.

The UAE has been more critical of Iran than other Arab states during the 2026 conflict, but it is also a major trading partner of Iran; Iran is the source of 30% of its imports (the largest source of imports, followed by China, Turkey and the European Union) and its third-largest export market (behind China and Iraq).

As noted in BBC Monitoring analysis, the UAE also plays an important role in facilitating Iranian access to global markets and finance. The US already has many sanctions in place against UAE-based entities for facilitating Iranian transactions.

The Wall Street Journal reports the UAE’s action against Iran may be phased in, beginning with shipping and extending to other sectors if Iranian attacks do not end.

### Iranian response

Major General Razaei said Iran would treat it as an “act of war” if any state supported the new US sanctions. The Economy Minister, Ali Madanizadeh, noted that neither Russia nor China had accepted the US measures.

---

## Related Links

Research relating to the US-Israel attacks on Iran, Iranian counterstrikes and the response of the UK and others.

(Links in original document not reproduced here.)

## Live News Feed Collection and Storage
Currently only from MoneyControl

In [11]:
_mc_links = mos1.get_mc_links("https://www.moneycontrol.com/world/", "/world/")

/world/ links located : 61


In [12]:
stories = mos1.get_stories(_mc_links)

Processed 10/61 | Valid stories: 8
Processed 20/61 | Valid stories: 16
Processed 30/61 | Valid stories: 22
Processed 40/61 | Valid stories: 26
Processed 50/61 | Valid stories: 33
Processed 60/61 | Valid stories: 42
Processed 61/61 | Valid stories: 43
----------------------------------------
Links examined : 61
Valid stories  : 43


In [13]:
# ------------------------------------------------------------------------
# Score stories for Middle East Oil Security relevance
# ------------------------------------------------------------------------

def score_MOS(article):

    text = (
        article["title"]
        + " "
        + article["content"]
    ).lower()

    score = 0

    high = [
        "strait of hormuz",
        "hormuz",
        "iran",
        "iranian",
        "crude oil",
        "oil price",
        "brent",
        "middle east",
        "persian gulf"
    ]

    geopolitical = [
        "united states",
        "u.s.",
        "israel",
        "missile",
        "drone",
        "air strike",
        "airstrike",
        "attack",
        "war",
        "military",
        "sanctions",
        "retaliation",
        "ceasefire"
    ]

    oil_security = [
        "tanker",
        "shipping",
        "oil supply",
        "oil exports",
        "oil production",
        "refinery",
        "opec",
        "opec+",
        "saudi arabia",
        "uae",
        "qatar",
        "gulf"
    ]

    india = [
        "india",
        "indian oil",
        "ioc",
        "bharat petroleum",
        "hindustan petroleum",
        "reliance industries",
        "oil import",
        "petroleum ministry"
    ]

    for k in high:
        if k in text:
            score += 4

    for k in geopolitical:
        if k in text:
            score += 2

    for k in oil_security:
        if k in text:
            score += 2

    for k in india:
        if k in text:
            score += 2

    return score

In [14]:
# ------------------------------------------------------------------------
# Score and sort stories by MOS relevance
# ------------------------------------------------------------------------

for s in stories:
    s["score"] = score_MOS(s)

stories = sorted(
    stories,
    key=lambda x: x["score"],
    reverse=True
)

for i, s in enumerate(stories, 1):
    print(f"{i:3}. [{s['score']:2}] {s['title']}")

  1. [28] India may be one of the biggest winners if Iran reopens Hormuz: What happens to crude, petrol and diesel prices?
  2. [26] Iran says US response on Hormuz plan expected 'hopefully' tomorrow after Qatar-mediated talks
  3. [24] Tehran awaits Trump’s response on war-end plan, says Araghchi after US-Iran talks via Qatar in New York
  4. [24] Asia may weather tight global diesel market better than Europe
  5. [24] Iran awaits Trump’s response, but threatens more attacks: What are Tehran’s conditions for reopening Strait of Hormuz?
  6. [22] US should not dare approach Strait of Hormuz: Mojtaba Khamenei says Iran ready to ‘humiliate’ enemies
  7. [20] USS George Washington in Arabian Sea after Khamenei’s Hormuz warning? CENTCOM shares video
  8. [18] ‘Hoax’: Trump denies offering Iran sanctions relief as Tehran rules out nuclear concessions
  9. [16] China’s biggest port project in Pakistan faces a reckoning: Why Gwadar has failed to become the trade hub it promised
 10. [16] ‘Ira

In [15]:
# ------------------------------------------------------------------------
# Inspect top MOS-ranked stories
# ------------------------------------------------------------------------

for i, s in enumerate(stories[:15], 1):

    print("=" * 80)
    print(f"{i}. SCORE: {s['score']}")
    print(s["title"])
    print()
    print(s["content"])
    print()

1. SCORE: 28
India may be one of the biggest winners if Iran reopens Hormuz: What happens to crude, petrol and diesel prices?

A reopening of the Strait of Hormuz could provide India with an immediate reduction in energy and shipping pressures, particularly at a time when Indian refiners are dealing with tighter Russian crude supplies, higher Gulf crude premiums and sharply elevated tanker costs. The strategic waterway has been at the centre of the US-Iran conflict since fighting began in late February. Before the disruption, roughly one-fifth of the world's oil and LNG shipments moved through the strait. For India, the route is particularly important because a substantial portion of its crude and LNG supplies originate in the Gulf. India imports around 90% of the crude oil it consumes. The country has responded to the disruption by widening its sourcing network, but the shift has come with higher freight, insurance and procurement costs. A sustained reopening of Hormuz would therefore

In [20]:
live_file = mos1.CreateLiveFeed(stories, 4)

Stories selected : 25
File created     : MOS-MC-LiveFeed_260929_1924_IST.txt


In [22]:
!ls -l "{WORLDFACTS}/NEWS"

total 93
-rw------- 1 root root 43536 Sep 29 10:05 MOS-MC-LiveFeed_260929_1535_IST.txt
-rw------- 1 root root 50844 Sep 29 13:50 MOS-MC-LiveFeed_260929_1920_IST.txt


In [23]:
!rm "{WORLDFACTS}"/NEWS/*.txt

In [24]:
!ls -l "{WORLDFACTS}"
NEWS_DIR = f"{WORLDFACTS}/NEWS"

!ls -l "{WORLDFACTS}/NEWS"
!ls -l "{WORLDFACTS}/MD"

total 12
drwx------ 2 root root 4096 Sep 29 13:47 MD
drwx------ 2 root root 4096 Sep 29 13:56 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF
total 0
total 230
-rw------- 1 root root  7772 Sep 29 13:39 IranData_260929.md
-rw------- 1 root root  6590 Sep 29 13:39 MissilesofIranMissil_260929.md
-rw------- 1 root root  8393 Sep 29 13:40 OilMarketReportSepte_260929.md
-rw------- 1 root root 12277 Sep 29 13:40 PressReleasePagePres_260929.md
-rw------- 1 root root  4290 Sep 29 13:43 ShortTermEnergyOutlo_260929.md
-rw------- 1 root root 77199 Sep 29 13:43 SIPRISummary_260929.md
-rw------- 1 root root 40618 Sep 29 13:45 SnapshotofIndiasOilG_260929.md
-rw------- 1 root root 18052 Sep 29 13:45 UKCommonsIranin2026S_260929.md
-rw------- 1 root root 57430 Sep 29 13:47 USCongressR488873_260929.md


In [25]:
!mv "{live_file}" "{NEWS_DIR}/"

In [26]:
!ls -l "{MD_DIR}"
!ls -l "{NEWS_DIR}"

total 230
-rw------- 1 root root  7772 Sep 29 13:39 IranData_260929.md
-rw------- 1 root root  6590 Sep 29 13:39 MissilesofIranMissil_260929.md
-rw------- 1 root root  8393 Sep 29 13:40 OilMarketReportSepte_260929.md
-rw------- 1 root root 12277 Sep 29 13:40 PressReleasePagePres_260929.md
-rw------- 1 root root  4290 Sep 29 13:43 ShortTermEnergyOutlo_260929.md
-rw------- 1 root root 77199 Sep 29 13:43 SIPRISummary_260929.md
-rw------- 1 root root 40618 Sep 29 13:45 SnapshotofIndiasOilG_260929.md
-rw------- 1 root root 18052 Sep 29 13:45 UKCommonsIranin2026S_260929.md
-rw------- 1 root root 57430 Sep 29 13:47 USCongressR488873_260929.md
total 50
-rw------- 1 root root 50844 Sep 29 13:54 MOS-MC-LiveFeed_260929_1924_IST.txt


In [ ]:
#Check balance
# https://platform.openai.com/usage?utm_source=chatgpt.com